# SDXL Optimized — Deployment Demo

This notebook demonstrates:
1. **LitServe API** — as requested in the assignment
2. **Gradio interactive UI** — for live comparison of presets

**Runtime: NVIDIA A100 GPU** (Colab Pro+)

In [ ]:
# === Setup ===
import os
os.chdir('/content')
if not os.path.exists('/content/sdxl-optimization/src'):
    !git clone https://github.com/rajat116/sdxl-optimization.git
os.chdir('/content/sdxl-optimization')
!pip install -q -e '.[dev]'
!pip install -q DeepCache 'litserve==0.2.5' 'gradio>=4.0.0'
print('\n✅ Setup complete')

## Part 1: LitServe API

In [ ]:
import subprocess, requests, time, base64, json
from PIL import Image
from io import BytesIO

!kill $(lsof -t -i:8000) 2>/dev/null || true
time.sleep(2)

proc = subprocess.Popen(
    ['python', 'server/serve.py', '--preset', 'speed', '--port', '8000'],
    stdout=open('/dev/null', 'w'),
    stderr=open('/tmp/server.log', 'w'),
)

print('Starting LitServe server with speed preset...')
print('Loading model (1-3 min on A100)...\n')

server_ready = False
for i in range(480):
    if proc.poll() is not None:
        print('❌ Crashed:'); print(open('/tmp/server.log').read()[-2000:]); break
    try:
        r = requests.post('http://localhost:8000/predict',
                          json={'prompt': 'test', 'height': 64, 'width': 64}, timeout=3)
        if r.status_code == 200:
            print(f'✅ Server ready after {i+1}s!')
            server_ready = True; break
    except: pass
    time.sleep(1)
    if i % 60 == 59:
        lines = open('/tmp/server.log').readlines()
        last = lines[-1].strip()[:100] if lines else '...'
        print(f'  Loading... ({i+1}s) — {last}')

In [ ]:
# === API Test: curl-equivalent ===
print('━' * 60)
print('  LitServe API — POST /predict')
print('━' * 60)

payload = {'prompt': 'a photo of an astronaut riding a horse on mars, cinematic lighting', 'seed': 42}

print(f'\ncurl -X POST http://localhost:8000/predict \\\')
print(f'  -H "Content-Type: application/json" \\\')
print(f'  -d \'{json.dumps(payload)}\'\n')

response = requests.post('http://localhost:8000/predict', json=payload, timeout=120)
data = response.json()

print(f'Status:  {response.status_code}')
print(f'Latency: {data["latency_s"]}s')
print(f'Preset:  {data["preset"]}')
print(f'Steps:   {data["steps"]}')
print(f'Image:   {len(data["image_base64"])} chars (base64 PNG)')

img = Image.open(BytesIO(base64.b64decode(data['image_base64'])))
display(img)

In [ ]:
# === Batch test: 4 prompts through API ===
import matplotlib.pyplot as plt

prompts = [
    'a photo of an astronaut riding a horse on mars, cinematic lighting',
    'a beautiful sunset over a calm ocean with sailboats, oil painting',
    'a corgi wearing a top hat and monocle, portrait photography',
    'a futuristic cityscape at night with neon lights, cyberpunk',
]

imgs, lats = [], []
for i, p in enumerate(prompts):
    r = requests.post('http://localhost:8000/predict',
                      json={'prompt': p, 'seed': 42}, timeout=120)
    d = r.json()
    imgs.append(Image.open(BytesIO(base64.b64decode(d['image_base64']))))
    lats.append(d['latency_s'])
    print(f'  [{i+1}/4] {d["latency_s"]}s')

avg = sum(lats)/len(lats)
print(f'\n📊 Avg: {avg:.2f}s | Throughput: {60/avg:.0f} img/min | Speedup: {5.66/avg:.1f}×')

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, im, p, l in zip(axes, imgs, prompts, lats):
    ax.imshow(im); ax.set_title(f'{l}s\n{p[:35]}...', fontsize=9); ax.axis('off')
plt.suptitle(f'LitServe API — speed preset | {avg:.2f}s/img | {60/avg:.0f} img/min',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Stop LitServe — we need the GPU for Gradio
proc.terminate(); proc.wait()
print('✅ LitServe server stopped.')

import gc, torch
gc.collect(); torch.cuda.empty_cache()

## Part 2: Interactive Gradio Demo

This launches an interactive UI with a **public shareable link**.

Features:
- Pick any preset and generate
- See real-time metrics (latency, speedup, VRAM)
- Compare Speed vs Quality side by side
- Full benchmark results table

In [ ]:
import os
os.chdir('/content/sdxl-optimization')
!kill $(lsof -t -i:7860) 2>/dev/null || true
import time; time.sleep(2)
%run app/demo.py